In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
df = spark.read.format("delta").table("airspace_pulse.silver.flight_states_enriched")

In [0]:
df = df.select(
    col("time_position").alias("event_time"),
    col("icao24"),
    col("callsign"),
    col("origin_country"),
    col("longitude"),
    col("latitude"),
    col("baro_altitude"),
    col("on_ground"),
    col("velocity"),
    col("true_track"),
    col("vertical_rate"),
    col("squawk"),
    col("kafka_timestamp")
)

In [0]:

df = df.filter(
    (col("latitude").between(-90, 90)) &
    (col("longitude").between(-180, 180)) &
    (col("velocity") >= 0) &
    (col("icao24").isNotNull())
)
df= df.withColumn("airline_icao_prefix", col("callsign").substr(0, 3))
df = df.withColumn("speed_kmh", round(col("velocity")*3.6, 2))
df = df.withColumn("speed_knots", round(col("velocity")*1.94384, 2))
df = df.withColumn("vertical_rate_ft_min", round(col("vertical_rate")*196.85, 2))
df = df.withColumn("flight_status_type", when(col("on_ground") == True, "ON_GROUND").when(col("vertical_rate") > 1.5, "ASCENDING").when(col("vertical_rate") < -1.5, "DESCENDING").otherwise("CRUISE"))
df = df.withColumn("is_emergency", when(col("squawk").isin('7500', '7600', '7700'), True).otherwise(False))
df = df.withColumn("emergency_reason", when(col("squawk")=='7500', 'HIJACKING').when(col("squawk")=='7600', 'RADIO FAILURE').when(col("squawk")=='7700', 'GENERAL EMERGENCY'))
df = df.fillna({"baro_altitude": 0.0, "vertical_rate": 0.0})



In [0]:
display(df)

In [0]:
import os
os.listdir("/Volumes/airspace_pulse/bronze/static_data/")


In [0]:
a_df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option("quote", "'") \
    .option("escape", "'") \
    .option("multiLine", "true") \
    .load("/Volumes/airspace_pulse/bronze/static_data/aircraft/aircraft-database.csv")

print(a_df.count())



In [0]:
# a_df = a_df.select("'icao24'", "'registration'", "'manufacturerName'", "'model'", "'operator'")

display(a_df)